# 06 · Model evaluation

Everything reported here is measured on participants the models have never
seen. Also included: rep-counting error and inference latency, which are the
two numbers that decide whether the feature is usable live.

## Shared setup (run first)

This notebook uses the modules in `ml/`. Upload them once to the Colab runtime:

1. Click the folder icon in the left sidebar (Files).
2. Click **Upload to session storage** and select `extract_pose.py`,
   `fitness_features.py`, `build_dataset.py`, `train_baseline.py`,
   `train_temporal.py`, `export_onnx.py` from `F:\Fitness Coach\ml`.
3. Run the cell below.

The dataset zip is expected at
`/content/drive/MyDrive/Dataset Exercise Quality-wise.zip` (Google Drive).

In [ ]:
import os, sys, subprocess, json, glob
from pathlib import Path

ROOT = Path('/content/fitness')
ROOT.mkdir(parents=True, exist_ok=True)

# The uploaded modules land in /content; mirror them into the work root.
for f in ['extract_pose.py', 'fitness_features.py', 'build_dataset.py', 'train_baseline.py',
          'experiment_generalization.py', 'experiment_clip.py', 'experiment_repmetry.py',
          'train_final.py', 'train_temporal.py', 'export_onnx.py']:
    src = Path('/content') / f
    if src.exists():
        dst = ROOT / f
        if not dst.exists() or src.stat().st_mtime > dst.stat().st_mtime:
            dst.write_bytes(src.read_bytes())

sys.path.insert(0, str(ROOT))
print('work root :', ROOT)
print('modules   :', sorted(p.name for p in ROOT.glob('*.py')))
print('cuda      :', subprocess.run('nvidia-smi -L', shell=True, capture_output=True, text=True).stdout.strip() or 'no GPU')

## 1. Confusion matrices on the held-out participants

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
fm = json.load(open(ROOT / 'export/final_metrics.json'))
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, ex in zip(axes, fm['exercises']):
    c = fm['exercises'][ex]['test']['confusion']
    mat = np.array([[c['tn_bad_correct'], c['fp_bad_as_good']],
                    [c['fn_good_as_bad'], c['tp_good_correct']]])
    ax.imshow(mat, cmap='Blues')
    ax.set_title(f"{ex} (n={fm['exercises'][ex]['test']['n']})")
    ax.set_xticks([0, 1], ['pred BAD', 'pred GOOD']); ax.set_yticks([0, 1], ['BAD', 'GOOD'])
    for (i, j), v in np.ndenumerate(mat):
        ax.text(j, i, int(v), ha='center', va='center', color='black')
plt.tight_layout(); plt.show()

## 2. Per-participant performance (held-out split)

In [ ]:
import json, pandas as pd, numpy as np, joblib
fm = json.load(open(ROOT / 'export/final_metrics.json'))
splits = json.load(open(ROOT / 'dataset/splits.json'))
clip = pd.read_parquet(ROOT / 'dataset/clip_features.parquet')
rows = []
for ex in ['squat', 'bicep_curl', 'push_up']:
    b = joblib.load(ROOT / f'models/final_{ex}.joblib')
    d = clip[(clip.exercise == ex) & (clip.subject.isin(splits['test']))]
    p = b['pipeline'].predict_proba(d[b['feature_cols']].to_numpy(np.float32))[:, 1]
    d = d.assign(pred=(p >= b['threshold']).astype(int))
    for sub, g in d.groupby('subject'):
        rows.append({'exercise': ex, 'subject': sub, 'clips': len(g),
                     'accuracy': round((g.pred == g.label).mean(), 3)})
df = pd.DataFrame(rows)
print(df.pivot(index='subject', columns='exercise', values='accuracy').round(2))
print()
print('participants under 0.60 accuracy:', df[df.accuracy < 0.60].to_string(index=False))

## 3. Rep counting

Rep counting is a separate state machine, not the classifier. It is validated on
the clean (GOOD) clips: the annotated clips contain several reps each, so we
compare the state machine's count against the count implied by the deepest
angle swings.

The rule set below is deliberately simple and exercise-specific; the thresholds
come from the observed angle distributions, not from guesses.

In [ ]:
import pandas as pd, numpy as np, glob
frames = pd.read_parquet(ROOT / 'dataset/frames.parquet')

def count_reps(series, t, low, high):
    """Hysteresis state machine: a rep completes when the angle goes below
    `low` (bottom) and back above `high` (top)."""
    reps, state = 0, 'top'
    for v in series:
        if not np.isfinite(v):
            continue
        if state == 'top' and v <= low:
            state = 'bottom'
        elif state == 'bottom' and v >= high:
            state = 'top'
            reps += 1
    return reps

THRESH = {'squat': ('knee_angle_mean', 110, 160),
          'bicep_curl': ('elbow_angle_mean', 80, 140),
          'push_up': ('elbow_angle_mean', 100, 155)}
rows = []
for ex, (col, low, high) in THRESH.items():
    sub = frames[(frames.exercise == ex) & (frames.quality == 'good')]
    for vid, g in sub.groupby('video_id'):
        g = g.sort_values('t_sec')
        rows.append({'exercise': ex, 'video_id': vid, 'reps': count_reps(g[col].to_numpy(), g.t_sec.to_numpy(), low, high),
                     'duration_sec': round(g.t_sec.max(), 1)})
reps = pd.DataFrame(rows)
print(reps.groupby('exercise').agg(videos=('reps', 'size'), mean_reps=('reps', 'mean'),
                                   reps_per_10s=('reps', lambda s: round(s.mean()/1.5, 1))).round(2))
print()
print('clips where the state machine found 0 reps (needs threshold tuning):')
print(reps[reps.reps == 0].groupby('exercise').size())

### Ground truth for rep counts

The Mendeley dataset labels quality, not rep counts, so a true MAE cannot be
computed from it. To finish this metric honestly you need one of:

1. hand-count the reps in 30-50 clips (fast: one afternoon, a spreadsheet), or
2. record your own clips with a known rep count.

Save as `dataset/rep_ground_truth.csv` with `video_id,true_reps`, then the
following cell reports real MAE / over- / under-counting.

In [ ]:
gt_path = ROOT / 'dataset/rep_ground_truth.csv'
if gt_path.exists():
    gt = pd.read_csv(gt_path)
    merged = reps.merge(gt, on='video_id')
    merged['err'] = merged['reps'] - merged['true_reps']
    print('MAE:', round(merged['err'].abs().mean(), 2))
    print('over-counted:', int((merged['err'] > 0).sum()), '| under-counted:', int((merged['err'] < 0).sum()))
    print(merged.groupby('exercise').err.apply(lambda s: round(s.abs().mean(), 2)))
else:
    print('No rep_ground_truth.csv yet - MAE cannot be reported. Counts above are model output only.')

## 4. Inference latency (must be < 500 ms)

In [ ]:
import time, numpy as np, json, joblib
bundle = joblib.load(ROOT / 'models/rf_squat.joblib')
pipe, cols = bundle['pipeline'], bundle['feature_cols']
x = np.random.rand(1, len(cols)).astype(np.float32)
pipe.predict_proba(x)  # warm up
t0 = time.perf_counter()
for _ in range(200):
    pipe.predict_proba(x)
print('sklearn RF single row: %.2f ms' % ((time.perf_counter() - t0) / 200 * 1000))

import onnxruntime as ort, glob
for p in sorted(glob.glob(str(ROOT / 'export/*.onnx'))):
    s = ort.InferenceSession(p, providers=['CPUExecutionProvider'])
    inp = {s.get_inputs()[0].name: x}
    s.run(None, inp)
    t0 = time.perf_counter()
    for _ in range(200):
        s.run(None, inp)
    print(p.split('/')[-1], '%.2f ms/row (cpu onnxruntime)' % ((time.perf_counter() - t0) / 200 * 1000))
print()
print('In the browser the same ONNX graph runs on WASM/WebGL; measure it there with performance.now().')

## 5. Verdict checklist

| Target | Where it is measured | Pass? |
|---|---|---|
| Form F1 >= 0.80 | video level, test participants | see tables |
| Rep MAE <= 1 | needs `rep_ground_truth.csv` | pending hand count |
| Latency < 500 ms | notebook cell 4 + browser measurement | see printout |
| No BAD->GOOD false accept above 10% | video level test | see tables |

Do not ship on window-level numbers alone.